Section 1 : Importing Libraries

In [1]:
import pandas as pd
import numpy as np

from scipy.sparse import hstack, csr_matrix

from sklearn.model_selection import train_test_split

import joblib

Section 2 : Loading Feature Engineered Dataset

In [2]:

df = pd.read_csv(
    "../data/processed/feature_engineered_dataset.csv"
)

print(f"Dataset Shape : {df.shape}")

df.head()

Dataset Shape : (7419, 21)


,City,Listing Type,Field,Job Title,Company Name,Salary/Stipend,Job Description,Location,Posted Date,Experience Level,...,Real/Fake Job,combined_text,label,text_length,word_count,avg_word_length,keyword_risk_score,salary_risk_score,domain,domain_frequency
0,Mumbai,Full-time,Tech,Software Developer,Rao IIT Academy,"₹500,000 - ₹1,200,000",as a software developer at rao iit academy you...,"Mumbai, Maharashtra",2019-05-17T21:02:37Z,Entry Level (0-2 years),...,Real Job,software developer rao iit academy tech entry ...,0,550,89,5.191011,0,1,www.adzuna.in,6553
1,Mumbai,Full-time,Tech,Software Developer,Cere Labs,"₹300,000 - ₹400,000",about us cere labs is a mumbai based company w...,"Mumbai, Maharashtra",2026-01-24T11:20:47Z,Entry Level (0-2 years),...,Real Job,software developer cere labs tech entry level ...,0,541,93,4.827957,0,1,www.adzuna.in,6553
2,Mumbai,Full-time,Tech,Software Developer,Image Media World,"₹500,000 - ₹700,000",job description software developer position ov...,"Mumbai, Maharashtra",2024-04-03T18:05:16Z,Entry Level (0-2 years),...,Real Job,software developer image media world tech entr...,0,549,78,6.051282,0,0,www.adzuna.in,6553
3,Mumbai,Full-time,Tech,Software Developer,Ion,Not specified,job summary writes new software makes modifica...,"Mumbai, Maharashtra",2026-05-17T06:47:16Z,Entry Level (0-2 years),...,Real Job,software developer ion tech entry level years ...,0,526,71,6.422535,0,0,www.adzuna.in,6553
4,Mumbai,Full-time,Tech,Software Developer,Etaash Consultants,"₹300,000 - ₹1,500,000",job opening in it company for angular develope...,"Mumbai, Maharashtra",2021-09-03T01:56:01Z,Entry Level (0-2 years),...,Real Job,software developer etaash consultants tech ent...,0,537,79,5.810127,0,0,www.adzuna.in,6553


Section 3: Load TF-IDF Vectorizer & Transform Text

In [3]:
# Load saved TF-IDF vectorizer

tfidf = joblib.load(
    "../models/tfidf_vectorizer.pkl"
)

print("TF-IDF Vectorizer Loaded.")

TF-IDF Vectorizer Loaded.


In [4]:
# Transform Combined Text


X_tfidf = tfidf.transform(
    df["combined_text"]
)

print("TF-IDF Matrix Shape:", X_tfidf.shape)

TF-IDF Matrix Shape: (7419, 5000)


In [5]:
# Validation

print("Number of Samples :", X_tfidf.shape[0])
print("Number of Features:", X_tfidf.shape[1])

Number of Samples : 7419
Number of Features: 5000


Section 4 : Selecting Numerical Features

In [6]:
numerical_features = [
    "text_length",
    "word_count",
    "avg_word_length",
    "keyword_risk_score",
    "salary_risk_score",
    "domain_frequency"
]

X_numeric = df[numerical_features]

print("Numerical Feature Shape:", X_numeric.shape)

X_numeric.head()

Numerical Feature Shape: (7419, 6)


,text_length,word_count,avg_word_length,keyword_risk_score,salary_risk_score,domain_frequency
0,550,89,5.191011,0,1,6553
1,541,93,4.827957,0,1,6553
2,549,78,6.051282,0,0,6553
3,526,71,6.422535,0,0,6553
4,537,79,5.810127,0,0,6553


In [7]:
# Converting Numerical Features to Sparse Matrix

X_numeric_sparse = csr_matrix(X_numeric.values)

print("Sparse Numerical Shape:", X_numeric_sparse.shape)

Sparse Numerical Shape: (7419, 6)


Section 5: Create Final Feature Matrix

In [8]:
# Combine TF-IDF and Numerical Features

X = hstack([
    X_tfidf,
    X_numeric_sparse
])

print("Final Feature Matrix Shape:", X.shape)

Final Feature Matrix Shape: (7419, 5006)


In [9]:
print(f"Total Samples  : {X.shape[0]}")
print(f"Total Features : {X.shape[1]}")

Total Samples  : 7419
Total Features : 5006


Section 6: Create Target Variable (y)

In [10]:
y = df["label"]

print("Target Shape :", y.shape)

print("\nTarget Distribution:\n")
print(y.value_counts())

Target Shape : (7419,)

Target Distribution:

label
0    6532
1     887
Name: count, dtype: int64


In [11]:
# Check for missing labels

assert y.isnull().sum() == 0, "Missing labels found!"

print("Target variable validated.")

Target variable validated.


Section-7: Train-Test Split

In [12]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Train-Test Split Completed")

Train-Test Split Completed


In [13]:
print(f"X_train Shape : {X_train.shape}")
print(f"X_test Shape  : {X_test.shape}")

print()

print("y_train Distribution")
print(y_train.value_counts())

print()

print("y_test Distribution")
print(y_test.value_counts())

X_train Shape : (5935, 5006)
X_test Shape  : (1484, 5006)

y_train Distribution
label
0    5225
1     710
Name: count, dtype: int64

y_test Distribution
label
0    1307
1     177
Name: count, dtype: int64


Section-8: Save Prepared Data

In [14]:
from pathlib import Path
from scipy.sparse import save_npz

# Create output directory
output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

# Save sparse matrices
save_npz(output_dir / "X_train.npz", X_train)
save_npz(output_dir / "X_test.npz", X_test)

# Save target variables
y_train.to_csv(output_dir / "y_train.csv", index=False)
y_test.to_csv(output_dir / "y_test.csv", index=False)

print("Prepared data saved.")

Prepared data saved.


Section 9 : Final Validation

In [15]:
print(f"X_train : {X_train.shape}")
print(f"X_test  : {X_test.shape}")

print(f"y_train : {y_train.shape}")
print(f"y_test  : {y_test.shape}")

print("\n Model Preparation Completed.")

X_train : (5935, 5006)
X_test  : (1484, 5006)
y_train : (5935,)
y_test  : (1484,)

 Model Preparation Completed.


In [17]:
# Check duplicate combined_text

duplicate_count = df["combined_text"].duplicated().sum()

print("Duplicate Combined Text:", duplicate_count)

Duplicate Combined Text: 74


In [16]:
# Convert sparse split indices back to dataframe split

X_train_df, X_test_df = train_test_split(
    df,
    test_size=0.20,
    random_state=42,
    stratify=df["label"]
)

train_texts = set(X_train_df["combined_text"])
test_texts = set(X_test_df["combined_text"])

common_texts = train_texts.intersection(test_texts)

print("Common Texts in Train & Test:", len(common_texts))

Common Texts in Train & Test: 20


In [18]:
# Check if target accidentally exists in features

print(df.columns.tolist())

['City', 'Listing Type', 'Field', 'Job Title', 'Company Name', 'Salary/Stipend', 'Job Description', 'Location', 'Posted Date', 'Experience Level', 'Application Link', 'Real/Fake Job', 'combined_text', 'label', 'text_length', 'word_count', 'avg_word_length', 'keyword_risk_score', 'salary_risk_score', 'domain', 'domain_frequency']


In [19]:
tfidf.fit_transform(df["combined_text"])

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 376589 stored elements and shape (7419, 5000)>